[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barmag/fanous-llm-lens/blob/main/notebooks/education/stage2_dash_skip_trigram_reference.ipynb)

In [ ]:
# Setup: install deps + fetch the shared helper on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens tokenizers huggingface_hub plotly
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
import tiny
import torch
import plotly.graph_objects as go

torch.manual_seed(42)
print("device:", tiny.device())

<div dir="rtl">

# المرحلة ٢داش: طبقة انتباه واحدة = خليط من bigram و skip-trigram (على عربي حقيقي)

في المرحلة ٢أ ضفنا كتلة انتباه واحدة وشفنا **شكل** الانتباه. هنا بنوصل للنتيجة الأساسية في ورقة *A Mathematical Framework for Transformer Circuits* (Elhage et al., 2021):

> **محوّل بطبقة واحدة، انتباه بس (من غير MLP)، بيتفكّك بالظبط لمجموع:**
> - **مسار مباشر (bigram):** بيتنبأ من التوكن الحالي بس — من غير سياق.
> - **دوائر skip-trigram (رأس لكل رأس):** بتبص لتوكن أقدم في السياق وتنسخ منه معلومة.

والخليط البسيط ده **معبّر بشكل مفاجئ**. عشان الدوائر تبان **مقروءة** مش ضوضاء، الموديل هنا **مدرّب على نطاق محترم**: ٨ رؤوس، d_model=512، على ~٣٤٠ مليون توكن عربي (فصحى أساسًا + مصري). درّبناه مرة واحدة بسكربت `train_stage2dash.py`؛ النوتبوك ده بيحمّل النتيجة ويفكّكها.

**حدود التجربة:** الكوربَس فصحى-أغلب (المصري نادر)، وبذرة واحدة. التفكيك نفسه **مظبوط رياضيًا** لأي موديل من النوع ده.

</div>

<div dir="ltr">

# Stage 2dash: one attention layer = a bigram + skip-trigram ensemble (on real Arabic)

The central result of *A Mathematical Framework for Transformer Circuits* (Elhage et al., 2021):

> **A one-layer attention-only transformer (no MLP) decomposes *exactly* into a sum of:**
> - **a direct path (bigram):** predicts from the current token alone — no context;
> - **skip-trigram circuits (one per head):** look back at an earlier token and copy from it.

This simple ensemble is **surprisingly expressive**. So the circuits read as signal rather than noise, the model here is trained at a **respectable scale**: 8 heads, d_model=512, on ~340M Arabic tokens (mostly MSA + some Masri). It was trained once by `train_stage2dash.py`; this notebook **loads** the result and decomposes it.

**Limits:** the corpus is MSA-heavy (Masri is scarce) and it's a single seed. The decomposition itself is **mathematically exact** for any such model.

</div>

<div dir="rtl">

## ١. التفكيك بالظبط · The exact decomposition

مخرجات الموديل (logits) بتساوي بالظبط:

$$\text{logits} = \underbrace{x\,W_U}_{\text{bigram}} \;+\; \underbrace{\sum_{\text{heads, positions}} a_{\text{src}}\;(x_{\text{src}}\,W_V W_O W_U)}_{\text{skip-trigram}}$$

الموديل **من غير LayerNorm** و**shortformer** (الموضع في مسار QK بس)، فالتفكيك يطلع مظبوط بالظبط والمسار المباشر يفضل bigram حقيقي.

</div>

<div dir="ltr">

## 1. The exact decomposition

The model's logits equal, exactly, a **bigram direct path** `x·W_U` plus a sum of per-head
**skip-trigram** terms `aₛ·(xₛ·W_V·W_O·W_U)`. The model is **LayerNorm-free** with a
**shortformer** positional embedding (position in the QK path only), so the split is exact
and the direct path stays a true bigram.

</div>

<div dir="rtl">

## ٢. نحمّل الموديل والمُجزّئ · Load the model & tokenizer

نحمّل نقطة الحفظ المدرّبة (محليًا، وإلا من Hugging Face Hub) — موديل بطبقة واحدة، انتباه بس، LN-free + shortformer، مع BPE عربي صغير (~١٢ ألف). للاختبار السريع في الـ CI نبني نسخة مصغّرة (FORCE_TINY).

</div>

In [ ]:
import os

CKPT_DIR = os.path.join(os.path.dirname(os.path.abspath(tiny.__file__)), "checkpoints", "stage2dash")
HF_REPO = "yassermakram/fanous-stage2dash-attn-only-1l"   # push once from train_stage2dash.py

# A small inline Arabic paragraph used for all eval/forward passes (no network needed,
# in-vocab for the trained model). Masri + MSA mix.
EVAL_TEXT = (
    "القطة بتاكل السمك والولد بيشرب اللبن في البيت. "
    "الجو حلو النهارده واحنا رايحين نتمشى في الشارع. "
    "المدينة كبيرة وفيها ناس كتير بتروح وتيجي كل يوم. "
    "هو قال إنه هيسافر بكرة بدري عشان الشغل المهم."
)

def _model_from_ckpt(ckpt):
    c = ckpt["config"]
    model = tiny.make_tiny_model(
        n_layers=c["n_layers"], n_heads=c["n_heads"], d_vocab=c["d_vocab"],
        n_ctx=c["n_ctx"], d_model=c["d_model"], attn_only=c["attn_only"],
        normalization_type=c["normalization_type"],
        positional_embedding_type=c["positional_embedding_type"])
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model

def _load_real():
    from tokenizers import Tokenizer
    tpath, mpath = os.path.join(CKPT_DIR, "tokenizer.json"), os.path.join(CKPT_DIR, "model.pt")
    if not (os.path.exists(tpath) and os.path.exists(mpath)):
        try:
            from huggingface_hub import hf_hub_download
            tpath = hf_hub_download(HF_REPO, "tokenizer.json")
            mpath = hf_hub_download(HF_REPO, "model.pt")
        except Exception as e:
            raise FileNotFoundError(
                f"No local checkpoint in {CKPT_DIR} and HF repo {HF_REPO!r} is unavailable "
                f"({type(e).__name__}). Either train it once with `python train_stage2dash.py` "
                f"(writes the checkpoint locally), set HF_REPO to a pushed model "
                f"(train_stage2dash.py --push-hub --hf-repo <you>/...), or set FORCE_TINY=True "
                f"for a quick structural demo on a tiny random model.") from e
    ckpt = torch.load(mpath, map_location=tiny.device(), weights_only=False)  # carries a config dict
    return _model_from_ckpt(ckpt), Tokenizer.from_file(tpath)

def _build_tiny():
    # Fast, network-free stand-in for CI: a tiny BPE on EVAL_TEXT + a tiny random model.
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFKC()
    tok.pre_tokenizer = pre_tokenizers.Whitespace()
    tok.train_from_iterator([EVAL_TEXT] * 20,
                            trainers.BpeTrainer(vocab_size=200, special_tokens=["[UNK]"]))
    model = tiny.make_tiny_model(n_layers=1, n_heads=2, d_vocab=tok.get_vocab_size(),
        n_ctx=32, d_model=64, attn_only=True,
        normalization_type=None, positional_embedding_type="shortformer")
    model.eval()
    return model, tok

def load_model_and_tokenizer():
    return _build_tiny() if globals().get("FORCE_TINY") else _load_real()

model, tok = load_model_and_tokenizer()
VOCAB = tok.get_vocab_size()
id_to_str = {i: (tok.id_to_token(i) or "[?]") for i in range(VOCAB)}
def encode(text):
    return tok.encode(text).ids

print(f"layers=1 heads={model.cfg.n_heads} d_model={model.cfg.d_model} "
      f"n_ctx={model.cfg.n_ctx} vocab={VOCAB}")

<div dir="rtl">

## ٣. نثبت التفكيك بإيدينا · The decomposition, by hand

نفكّك الـ logits لتتابع حقيقي لجزئين — **المباشر** (تضمين × `W_U`) و**الـ skip-trigram** (مخرج الانتباه × `W_U`) — ومجموعهم لازم يطابق مخرج الموديل بالظبط: **الفرق ~صفر**.

</div>

In [ ]:
ids = torch.tensor([encode(EVAL_TEXT)[: model.cfg.n_ctx]]).to(tiny.device())
logits, cache = model.run_with_cache(ids)

direct = cache["resid_pre", 0] @ model.W_U + model.b_U   # bigram path       <- (1, ctx, V)
skip   = cache["attn_out", 0] @ model.W_U                # skip-trigram path  <- (1, ctx, V)
print("max |(direct + skip) - logits|:", float((direct + skip - logits).abs().max()))  # ~0

# how big is the skip-trigram contribution relative to the bigram skeleton?
print("||skip|| / ||direct|| (logit space):", round(float(skip.norm() / direct.norm()), 2))

<div dir="rtl">

## ٤. دائرة الـ bigram (المسار المباشر) · The bigram circuit

`W_E W_U` هي **هيكل الـ bigram**: لكل توكن، إيه التوكنز اللي بيرجّحها مباشرة من غير سياق. التوكنز الفصحى المتكررة بتطلع **متلازمات نظيفة** (على الرغم، من أجل، البيت الأبيض). التوكنز المصري (زي "اللي") **ضعيفة** لأن الكوربَس فصحى-أغلب — وده بنفسه ملاحظة عن ندرة بيانات اللهجة.

</div>

<div dir="ltr">

## 4. The bigram circuit

`W_E W_U` is the **bigram skeleton**: for each token, what it directly favours next, with no
context. Frequent MSA tokens give **clean collocations**; Masri tokens (e.g. "اللي") are
**weak** because the corpus is MSA-heavy — itself an observation about dialect-data scarcity.

</div>

In [ ]:
bigram = (model.W_E @ model.W_U).detach().cpu()   # (V, V): current token -> next token

def bigram_top(word, k=6):
    seq = encode(word)
    if not seq:
        return f"{word!r}: empty"
    tid = seq[-1]
    _, idx = torch.topk(bigram[tid], k)
    return f"{id_to_str[tid]:>8} ->  " + " · ".join(id_to_str.get(int(j), "[?]") for j in idx)

print("MSA (well-trained) — crisp collocations:")
for w in ["في", "من", "على", "كان", "البيت"]:
    print(" ", bigram_top(w))
print("\nMasri (data-starved here; corpus is MSA-heavy) — weaker / fragmented:")
for w in ["اللي", "عايز", "دلوقتي"]:
    print(" ", bigram_top(w))

<div dir="rtl">

## ٥. دوائر الـ skip-trigram (QK + OV) · The skip-trigram circuits

كل واحد من الـ ٨ رؤوس = جدول skip-trigram، بيتبني من دائرتين:
- **QK** = `W_E W_Q W_K^T W_E^T`: لتوكن وجهة، **هنبص على مين** (الـ source)؟
- **OV** = `W_E W_V W_O W_U`: الـ source ده، **إيه الناتج اللي بيرجّحه**؟

كتير من الرؤوس **دوائر نسخ (copy)**: لما الرأس يبص على توكن، الـ OV بيرجّع **نفس التوكن** للناتج (قطر مصفوفة OV عالي) — وده أبسط شكل لـ "نسخ من السياق" في طبقة واحدة. نطبع التوكن اللي كل رأس بينسخه أقوى، ومثال ثلاثي كامل `[source … destination → output]`. (بنقصر البحث على التوكنز المتكررة عشان التضمينات النادرة ماتسيطرش.)

</div>

<div dir="ltr">

## 5. The skip-trigram circuits

Each of the 8 heads is a skip-trigram table, built from two circuits — **QK**
(`W_E W_Q W_Kᵀ W_Eᵀ`: which source to attend to) and **OV** (`W_E W_V W_O W_U`: what the
attended source promotes). Many heads are **copy circuits**: attending to a token makes the
OV promote *that same token* (a high OV diagonal) — the simplest form of in-context copying a
one-layer model can do. We print the token each head most strongly copies, plus one full
triple. (Search is restricted to frequent tokens so rare high-norm embeddings don't dominate.)

</div>

In [ ]:
W_E, W_U = model.W_E.detach().cpu(), model.W_U.detach().cpu()
FREQ = min(2500, VOCAB)   # restrict to frequent tokens (rare embeddings have huge norms)

def head_circuits(h):
    QK = W_E @ model.W_Q[0, h].detach().cpu() @ model.W_K[0, h].detach().cpu().T @ W_E.T  # dst x src
    OV = W_E @ model.W_V[0, h].detach().cpu() @ model.W_O[0, h].detach().cpu() @ W_U      # src x out
    return QK, OV

print("Each head is a skip-trigram table. The token each head most strongly COPIES")
print("(attend to it -> the OV promotes it back; high OV diagonal):")
for h in range(model.cfg.n_heads):
    QK, OV = head_circuits(h)
    s = int(torch.topk(OV.diagonal()[:FREQ], 1).indices[0])   # strongest self-copy source
    o = int(torch.topk(OV[s, :FREQ], 1).indices[0])           # what attending to s promotes
    tag = "(copy)" if o == s else f"-> {id_to_str.get(o, '[?]')}"
    print(f"  head {h}: attend to {id_to_str.get(s, '[?]'):>12}  =>  promote {id_to_str.get(s, '[?]')} {tag}")

# one full skip-trigram triple [source … destination -> output] for head 0
QK, OV = head_circuits(0)
s = int(torch.topk(OV.diagonal()[:FREQ], 1).indices[0])
d = int(torch.topk(QK[:FREQ, s], 1).indices[0])   # a destination that routes attention to s
o = int(torch.topk(OV[s, :FREQ], 1).indices[0])
print(f"\nexample triple (head 0): [{id_to_str.get(s)} … {id_to_str.get(d)} -> {id_to_str.get(o)}]")

<div dir="rtl">

## ٦. خريطة الانتباه · Attention heatmap

دائرة الـ QK بتقرر نمط الانتباه. نشوفه لرأس واحد على جملة حقيقية: كل صف (وجهة) بيوزّع انتباهه على التوكنز اللي قبله. (بنعرض من اليمين للشمال زي العربي.)

</div>

In [ ]:
sent = "هو قال إنه هيسافر بكرة بدري عشان الشغل"
sids = encode(sent)[: model.cfg.n_ctx]
str_toks = [id_to_str.get(i, "[?]") for i in sids]
_, hcache = model.run_with_cache(torch.tensor([sids]).to(tiny.device()))
pattern = hcache["pattern", 0][0, 0].detach().cpu().numpy()  # head 0  <- (seq, seq)

fig = go.Figure(go.Heatmap(z=pattern, x=str_toks, y=str_toks, colorscale="Blues"))
fig.update_layout(title="Head 0 attention — " + sent, xaxis=dict(side="top"), height=460, width=560)
fig.update_xaxes(autorange="reversed")  # RTL
fig.update_yaxes(autorange="reversed")
fig.show()

<div dir="rtl">

## ٧. معبّر بشكل مفاجئ · Surprisingly expressive

المسار المباشر (bigram) **أعمى للسياق** — بيعتمد على التوكن الحالي بس. الـ skip-trigram هو اللي بيضيف الحساسية للسياق. نقيس على تتابعات حقيقية **قد إيه الـ skip-trigram بيعيد تشكيل** التوزيع بعيد عن الـ bigram، وكام مرة بيغيّر التوكن رقم ١.

</div>

In [ ]:
W = min(16, model.cfg.n_ctx)
flat = encode(EVAL_TEXT)
wins = [flat[i:i + W] for i in range(0, len(flat) - W, 4)][:32]
batch = torch.tensor(wins).to(tiny.device())

logits2, cache2 = model.run_with_cache(batch)
direct2 = cache2["resid_pre", 0] @ model.W_U + model.b_U      # bigram-only logits
p_full, p_dir = torch.softmax(logits2, -1), torch.softmax(direct2, -1)

reshape = float((p_full - p_dir).abs().max(-1).values.mean())          # avg distribution shift
changed = float((p_full.argmax(-1) != p_dir.argmax(-1)).float().mean())  # top-1 changed
print("The bigram path depends only on the current token (context-blind by construction).")
print(f"skip-trigram reshapes the next-token distribution by avg {reshape:.3f}")
print(f"and changes the top-1 prediction at {changed * 100:.1f}% of positions")
print("=> the ensemble's context-sensitivity is exactly the skip-trigram term.")

<div dir="rtl">

## ٨. الخلاصة · Recap

- موديل بطبقة واحدة، انتباه بس، مدرّب على ~٣٤٠ مليون توكن عربي، بيتفكّك **بالظبط** لـ **مسار bigram** + **دوائر skip-trigram** (٨ رؤوس) — الفرق ~صفر.
- **الـ bigram** (`W_E W_U`) = التوقع المباشر بلا سياق.
- **الـ skip-trigram** (`W_E W_QK W_E` للاختيار، `W_E W_OV W_U` للنسخ) بيبص لتوكن أقدم وينسخ منه — ودلوقتي الثلاثيات **مقروءة** لأن الموديل على نطاق محترم.
- الخليط **معبّر**: المباشر أعمى للسياق، والموديل الكامل بيعيد تشكيل التوقع — والفرق كله من الـ skip-trigram.

ده محور *A Mathematical Framework for Transformer Circuits* (2021)، متحقّق على المصري/الفصحى.

</div>

<div dir="ltr">

## 8. Recap

- A one-layer attention-only model trained on ~340M Arabic tokens decomposes **exactly**
  into a **bigram direct path** + **skip-trigram circuits** (8 heads) — difference ≈ 0.
- The **bigram** (`W_E W_U`) is the context-free direct prediction.
- The **skip-trigram** (`W_E W_QK W_E` selects, `W_E W_OV W_U` copies) looks back at an
  earlier token and copies from it — and at this scale the triples are **legible**.
- The ensemble is **expressive**: the direct path is context-blind, yet the full model
  reshapes its prediction — and that difference is entirely the skip-trigrams.

The thesis of *A Mathematical Framework for Transformer Circuits* (2021), on Masri/MSA.

</div>